# EXP-004 필터링 dev-small 확인

목적: 조건 필드가 2개 이상 맞은 문서를 앞으로 옮기는 필터링(EXP-004)이 동작하는지, 바뀐 질의가 조건 목록이 생긴 질의뿐인지, 보완할 점이 있는지 dev-small에서 본다. 판정은 dev-full·multi-doc으로만 한다(`docs/experiments/EXP-004.md`). 질의 원문은 출력하지 않는다(D-10).

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from rag.conditions import ConditionIndex
from rag.eval.metrics import gold_ranks, load_cfg, load_gold, min_docs, per_query, read_run
from rag.index import load_cfg as load_hydra

In [2]:
ROOT = Path.cwd().parent
LAYER = "dev-small"
RUNS = {
    "exp001": ROOT / "data/runs/kure-v1-fixed-512-64-hybrid-dev-small.csv",
    "exp003": ROOT / "data/runs/kure-v1-fixed-512-64-hybrid-cond-dev-small.csv",
    "exp004": ROOT / "data/runs/kure-v1-fixed-512-64-hybrid-cond-filter-dev-small.csv",
}
CONDITIONS = ROOT / "data/conditions/summary_q.jsonl"

## 순위 파일

`uv run python -m rag.search +exp=exp004`로 만들었다. EXP-003 순위도 바뀐 코드(`rrf`가 후보 전체를 내도록 바꿈)로 다시 만들었다. EXP-003 dev-small은 EXP-001과 순위가 모두 같았으므로(14_01) 다시 만든 파일도 같아야 한다.

In [3]:
runs = {name: read_run(path) for name, path in RUNS.items()}
order = {name: r.sort_values(["qid", "rank"]).groupby("qid")["doc_id"].apply(tuple)
         for name, r in runs.items()}
for name, run in runs.items():
    per_q = run.groupby("qid")["doc_id"].nunique()
    print(name, run.shape, "질의당 문서 수", per_q.min(), "~", per_q.max())
print("exp003 == exp001:", (order["exp003"] == order["exp001"]).all())

exp001 (10390, 3) 질의당 문서 수 10 ~ 10
exp003 (10390, 3) 질의당 문서 수 10 ~ 10
exp004 (10390, 3) 질의당 문서 수 10 ~ 10
exp003 == exp001: True


## 바뀐 질의

필터는 조건 목록이 생긴 질의(필드 2개 이상 맞은 문서가 있는 질의)에만 걸린다.

In [4]:
hcfg = load_hydra(["+exp=exp004"])
with CONDITIONS.open(encoding="utf-8") as f:
    conds = {r["qid"]: r["conditions"] for r in map(json.loads, f)}
small = set((ROOT / hcfg.paths.dev_small_docs).read_text(encoding="utf-8").split())
with (ROOT / hcfg.paths.corpus).open(encoding="utf-8") as f:
    docs = [d for d in map(json.loads, f) if d["doc_id"] in small]
idx = ConditionIndex(docs, hcfg.retriever.conditions.year_ref_offsets)
name_of = np.array([d["doc_id"] for d in docs])
min_fields = hcfg.retriever.conditions.min_fields
qids = order["exp003"].index
has_list = pd.Series({q: any(c >= min_fields for c in idx.match_counts(conds[q]).values())
                      for q in qids})
changed = order["exp003"] != order["exp004"]
pd.crosstab(has_list, changed, rownames=["조건 목록 있음"], colnames=["순위 바뀜"])

순위 바뀜,False
조건 목록 있음,
False,1039


## 지표 비교

In [5]:
cfg = load_cfg()
ev = cfg.eval
gold = load_gold(cfg, LAYER)
n_min = min_docs(ev.recall_ks, ev.mrr_k)
ranks = {name: gold_ranks(run, gold, n_min) for name, run in runs.items()}
scores = {name: per_query(r, ev.recall_ks, ev.mrr_k) for name, r in ranks.items()}
pd.DataFrame({name: s.mean() for name, s in scores.items()}).round(4)

,exp001,exp003,exp004
recall@1,0.9105,0.9105,0.9105
recall@5,0.9808,0.9808,0.9808
recall@10,0.9875,0.9875,0.9875
mrr@10,0.9438,0.9438,0.9438


## 조건 목록이 생긴 질의

질의마다 조건 필드, 필드 2개 이상 맞은 문서 수, 정답 문서가 맞은 필드 수, 정답 순위를 본다.

In [13]:
pos = {d: i for i, d in enumerate(name_of)}
cols = ["qid", "조건 필드", "2개 이상 맞은 문서 수", "정답이 맞은 필드 수",
        "exp003 정답 순위", "exp004 정답 순위"]
rows = []
for q in has_list[has_list].index:
    counts = idx.match_counts(conds[q])
    rows.append([q, ", ".join(sorted(conds[q])), sum(c >= min_fields for c in counts.values()),
                 counts.get(pos[gold.loc[q, "doc_id"]], 0), ranks["exp003"][q], ranks["exp004"][q]])
print("조건 목록이 생긴 dev-small 질의:", int(has_list.sum()))
pd.DataFrame(rows, columns=cols).set_index("qid")

조건 목록이 생긴 dev-small 질의: 0


,조건 필드,2개 이상 맞은 문서 수,정답이 맞은 필드 수,exp003 정답 순위,exp004 정답 순위
qid,,,,,


dev-small 질의 중 조건 필드가 2개 이상 뽑힌 질의는 3건이지만, dev-small 문서 범위에서 필드 2개 이상 맞는 문서가 없어 필터가 한 번도 걸리지 않는다. dev-small로는 필터 동작을 볼 수 없다.

## multi-doc 질의에서 필터가 걸리는 모양(정답·점수를 보지 않음)

코퍼스 전체 문서와 multi-doc 질의 조건만으로, 질의마다 필드 2개 이상 맞는 문서가 몇 개인지 센다. 10개 이상이면 상위 10개가 모두 조건에 맞는 문서로 채워진다. 정답 문서와 검색 점수는 보지 않는다(판정 전 설계 확인).

In [8]:
hcfg_full = load_hydra(["+exp=exp004", "index.scope=full", "search.queries=multidoc"])
with (ROOT / hcfg_full.paths.corpus).open(encoding="utf-8") as f:
    all_docs = list(map(json.loads, f))
idx_full = ConditionIndex(all_docs, hcfg_full.retriever.conditions.year_ref_offsets)
with (ROOT / hcfg_full.paths.conditions).open(encoding="utf-8") as f:
    md_conds = {r["qid"]: r["conditions"] for r in map(json.loads, f)}
n_match = pd.Series({q: sum(c >= min_fields for c in idx_full.match_counts(cd).values())
                     for q, cd in md_conds.items()})
n_match.describe().round(1)

count      600.0
mean      3270.9
std       3375.0
min          8.0
25%        931.2
50%       1488.0
75%       5454.2
max      14831.0
dtype: float64

In [9]:
bins = [-1, 0, 4, 9, 49, 99, 499, np.inf]
labels = ["0", "1~4", "5~9", "10~49", "50~99", "100~499", "500+"]
type_of = pd.Series({q: q.split("-")[1] for q in md_conds})
pd.crosstab(pd.cut(n_match, bins, labels=labels), type_of, margins=True)

col_0,conf,law,questioner,All
row_0,,,,
5~9,0,0,1,1
10~49,2,1,2,5
50~99,4,6,7,17
100~499,16,7,32,55
500+,211,138,173,522
All,233,152,215,600


필드 2개가 시간 정보(`date`와 `year_ref`)만으로 채워진 문서가 맞는 문서에 섞이는지 본다(PR #57 판단할 곳 1). 질의에 둘이 함께 있을 때만 생긴다.

In [10]:
both = [q for q, cd in md_conds.items() if "date" in cd and "year_ref" in cd]
time_only = {}
for q in both:
    cd = md_conds[q]
    full = idx_full.match_counts(cd)
    rest = idx_full.match_counts({k: v for k, v in cd.items() if k not in ("date", "year_ref")})
    time_only[q] = sum(1 for i, c in full.items() if c >= min_fields and rest.get(i, 0) == 0)
print("date·year_ref가 함께 있는 질의", len(both))
pd.Series(time_only).describe().round(1)

date·year_ref가 함께 있는 질의 54


count     54.0
mean      19.4
std       61.9
min        0.0
25%        0.0
50%        0.0
75%       22.0
max      442.0
dtype: float64

필드 2개 이상 맞는 문서가 질의당 중앙값 1,488개라 `min_fields` 2는 회의를 좁히지 못한다. 연도만 있는 날짜와 회의 종류(`소위원회`, `국정감사`)처럼 넓은 필드 둘이 맞으면 수천 문서가 된다. 질의마다 맞은 필드 수가 가장 많은 문서 수와, 뽑힌 필드가 모두 맞은 문서 수를 센다(정답을 보지 않음).

In [14]:
rows = []
for q, cd in md_conds.items():
    counts = idx_full.match_counts(cd)
    top = max(counts.values())
    rows.append({"qid": q, "유형": q.split("-")[1], "뽑힌 필드 수": len(cd),
                 "최다 일치 필드 수": top,
                 "최다 일치 문서 수": sum(c == top for c in counts.values()),
                 "모든 필드 일치 문서 수": sum(c == len(cd) for c in counts.values())})
shape = pd.DataFrame(rows).set_index("qid")
shape.groupby("유형")[["최다 일치 문서 수", "모든 필드 일치 문서 수"]].describe().round(1).T

유형                    conf    law  questioner
최다 일치 문서 수    count  233.0  152.0       215.0
              mean    12.2   11.1         4.1
              std      9.7   14.2         7.2
              min      1.0    1.0         1.0
              25%      5.0    2.0         2.0
              50%     12.0    4.5         2.0
              75%     16.0   17.0         3.0
              max     62.0  125.0        51.0
모든 필드 일치 문서 수 count  233.0  152.0       215.0
              mean    11.4    9.2         3.1
              std      9.9   10.7         4.2
              min      0.0    0.0         0.0
              25%      4.0    2.0         2.0
              50%     10.0    3.0         2.0
              75%     16.0   16.0         3.0
              max     62.0   45.0        47.0

In [15]:
zero = (shape["모든 필드 일치 문서 수"] == 0).groupby(shape["유형"]).sum()
print("모든 필드 일치 문서가 0개인 질의", zero.to_dict())
pd.crosstab(pd.cut(shape["최다 일치 문서 수"], bins, labels=labels), shape["유형"], margins=True)

모든 필드 일치 문서가 0개인 질의 {'conf': 12, 'law': 19, 'questioner': 5}


유형,conf,law,questioner,All
최다 일치 문서 수,,,,
1~4,51,76,191,318
5~9,50,15,13,78
10~49,127,60,10,197
50~99,5,0,1,6
100~499,0,1,0,1
All,233,152,215,600


## `filter: best`로 다시 실행

위 분포를 보고 사용자 결정(2026-10-12)으로 필터 대상을 "필드 2개 이상 맞은 문서"에서 "맞은 필드 수가 가장 많은 문서"로 바꿨다(질의당 중앙값 conf 12, law 4.5, questioner 2개). 정답과 검색 점수는 보지 않고 정했다. 같은 명령으로 dev-small 순위 파일을 다시 만들었다. dev-small에는 조건 목록이 생긴 질의가 없으므로 순위는 EXP-003과 같아야 한다.

In [16]:
runs["exp004"] = read_run(RUNS["exp004"])
order["exp004"] = runs["exp004"].sort_values(["qid", "rank"]).groupby("qid")["doc_id"].apply(tuple)
print("exp004 == exp003:", (order["exp004"] == order["exp003"]).all())

exp004 == exp003: True
